# Assignment3 - Green Patent Detection: Advanced Architectures (Option: Agentic-CrewAI llama3:4b)
> Suchanya Baiyam Business Data Science AAU
 - Investigate if advanced architectures produce better training data.
 - Must choose one of two paths: implementing a Multi-Agent System (MAS) to debate the claims, or Fine-Tuning an LLM (QLoRA) to learn the specific linguistic style of patent claims.
 - Then compare the performance from Baseline, Model1, this Model


Part A & B:

- Dataset: Load patents_50k_green.parquet (the balanced 50k sample).

- Uncertainty Sampling: Re-use the uncertainty scores u from  baseline model.

- Export: Select the same top 100 high-risk claims used in Assignment 2.


STEP01: Load libraries

In [ ]:
import pandas as pd
import numpy as np
import os

STEP02: Load dataset (Balanced patent 50K and reuse the u score from assignment 2 part B)

In [ ]:
df = pd.read_parquet("patent_50k_green.parquet")
df.shape

(50000, 4)

In [ ]:
df.head()

,doc_id,text,is_green_silver,split
402589,9669654,1. A paint-bucket mesh with a roller cleaning ...,0,train_silver
89394,8984304,1. A method for reducing power consumption in ...,1,train_silver
181025,9234489,1. A method for operating a temperature-limiti...,1,train_silver
25822,9001929,1. A method for a transmitter to transmit data...,0,train_silver
197203,8499387,"1. A treatment couch having a lying surface, a...",0,train_silver


In [ ]:
high_risk_100 = pd.read_csv("hitl_green_100.csv")
high_risk_100.shape

(100, 9)

In [ ]:
high_risk_100.head()

,doc_id,text,p_green,u,llm_green_suggested,llm_confidence,llm_rationale,is_green_human,notes
0,9315787,1. A DNA polymerase whose amino acid sequence ...,0.500019,0.999963,NaN,NaN,NaN,NaN,NaN
1,9334087,1. A standing pouch comprising: a first pack w...,0.499953,0.999907,NaN,NaN,NaN,NaN,NaN
2,9686855,1. A multilayer ceramic capacitor with interpo...,0.499926,0.999853,NaN,NaN,NaN,NaN,NaN
3,9551706,1. A method of detecting differences in densit...,0.499659,0.999317,NaN,NaN,NaN,NaN,NaN
4,9745777,1. An electronically controlled hatch system f...,0.500455,0.999091,NaN,NaN,NaN,NaN,NaN


Part C: Run CREW AI using ollama gemma 3:4b defines 3 roles with different propmt locally

Prompt using: Role-based
*   Advocate:
*   Skeptical:
*   judge:




In [ ]:
a3_agent_100 = pd.read_csv("A3_agent_labels_100_FINAL.csv")

In [ ]:
a3_agent_100.shape


(100, 5)

In [ ]:
a3_agent_100.head()

,doc_id,advocate_argument,skeptic_argument,is_green_tech,judge_rationale
0,9315787,The DNA polymerase patent represents a signifi...,Classifying this DNA polymerase solely under C...,1,The evidence presented by both the advocate an...
1,9334087,This patent claim describes a novel standing p...,The patent’s description focuses on a packagin...,1,While the advocate’s argument correctly identi...
2,9686855,"This patent claim, detailing a multilayer cera...",The patent’s claims regarding energy efficienc...,0,The arguments presented highlight a key tensio...
3,9551706,The patent describes a novel method for detect...,The patent’s description primarily outlines a ...,0,"The arguments presented highlight a valid, tho..."
4,9745777,An electronically controlled hatch system for ...,The patent claim describes a mechanical hatch ...,0,Both arguments highlight valid points. The adv...


In [ ]:
for i in range(3):
    row = a3_agent_100.iloc[i]

    print("="*80)
    print(f"DOC_ID: {row['doc_id']}")
    print("\nADVOCATE:")
    print(row['advocate_argument'])
    print("\nSKEPTIC:")
    print(row['skeptic_argument'])
    print("\nPREDICTION:", row['is_green_tech'])
    print("\nJUDGE RATIONALE:")
    print(row['judge_rationale'])
    print("="*80)
    print("\n\n")

DOC_ID: 9315787

ADVOCATE:
The DNA polymerase patent represents a significant step forward in green biotechnology – a move toward precision, efficiency, and reduced environmental impact. The targeted amino acid modifications, combined with the high degree of similarity to SEQ ID NO:38, demonstrate a deliberate effort to optimize an existing enzyme, reducing resource consumption and minimizing the risk of unintended consequences. This aligns perfectly with the core principles of CPC Y02, particularly its focus on biotechnological processes that minimize environmental impact and promote sustainable enzyme efficiency.

SKEPTIC:
Classifying this DNA polymerase solely under CPC Y02 risks overstating its green credentials. The claim’s reliance on a pre-existing, naturally occurring enzyme, coupled with the ‘at least 95% identical’ specification, suggests a refinement rather than a revolutionary advancement. The focus on optimization doesn’t inherently guarantee a reduced environmental impact

I noticed that when i run locally, i erase the text column. so i have to re-add it back

In [ ]:
a3_agent_100 = a3_agent_100.merge(
    high_risk_100[["doc_id", "text"]],
    on="doc_id",
    how="left"
)

In [ ]:
a3_agent_100.head()

,doc_id,advocate_argument,skeptic_argument,is_green_tech,judge_rationale,is_green_gold,human_notes,text
0,9315787,The DNA polymerase patent represents a signifi...,Classifying this DNA polymerase solely under C...,1,The evidence presented by both the advocate an...,None,,1. A DNA polymerase whose amino acid sequence ...
1,9334087,This patent claim describes a novel standing p...,The patent’s description focuses on a packagin...,1,While the advocate’s argument correctly identi...,None,,1. A standing pouch comprising: a first pack w...
2,9686855,"This patent claim, detailing a multilayer cera...",The patent’s claims regarding energy efficienc...,0,The arguments presented highlight a key tensio...,None,,1. A multilayer ceramic capacitor with interpo...
3,9551706,The patent describes a novel method for detect...,The patent’s description primarily outlines a ...,0,"The arguments presented highlight a valid, tho...",None,,1. A method of detecting differences in densit...
4,9745777,An electronically controlled hatch system for ...,The patent claim describes a mechanical hatch ...,0,Both arguments highlight valid points. The adv...,None,,1. An electronically controlled hatch system f...


Part D:Human Review & Final Integration

STEP01: HITL

Review the 100 claims + the AI rationale (from Agents or QLoRA). Create a final column is_green_gold based on your human judgment.

In [ ]:
from IPython.display import clear_output

In [ ]:
a3_agent_100["is_green_gold"] = None
a3_agent_100["human_notes"] = ""

In [ ]:
for i, row in a3_agent_100.iterrows():

    # skip already labeled
    if pd.notnull(row["is_green_gold"]):
        continue

    print("="*100)
    print(f"Reviewing index {i} ({i+1}/{len(a3_agent_100)})")
    print("="*100)

    print("\n📄 CLAIM TEXT:\n")
    print(row["text"])

    print("\n🤖 AGENT DEBATE:\n")

    print("ADVOCATE:")
    print(row["advocate_argument"])

    print("\nSKEPTIC:")
    print(row["skeptic_argument"])

    print("\nJUDGE DECISION:")
    print("Prediction:", row["is_green_tech"])
    print("Rationale:", row["judge_rationale"])

    print("\n------------------------------------")
    print("Press Enter to ACCEPT agent decision")
    print("Or type 0 / 1 to override")
    print("------------------------------------")

    user_input = input("Your decision: ").strip()

    if user_input == "":
        final_label = row["is_green_tech"]
    else:
        final_label = int(user_input)

    note = input("Optional note (press Enter to skip): ")

    a3_agent_100.loc[i, "is_green_gold"] = final_label
    a3_agent_100.loc[i, "human_notes"] = note

    # save progress
    a3_agent_100.to_csv("A3_agent_100_human_progress.csv", index=False)

print("Human review complete!")

Reviewing index 0 (1/100)

📄 CLAIM TEXT:

1. A DNA polymerase whose amino acid sequence is at least 95% identical to that of SEQ ID NO:38, but differs from SEQ ID NO:38 at a position corresponding to F749 of SEQ ID NO:38 and furthermore differs from SEQ ID NO:38 by at least two additional amino acids at positions corresponding to at least two of A61, K346, 5357, E507 or I707 of SEQ ID NO:38.

🤖 AGENT DEBATE:

ADVOCATE:
The DNA polymerase patent represents a significant step forward in green biotechnology – a move toward precision, efficiency, and reduced environmental impact. The targeted amino acid modifications, combined with the high degree of similarity to SEQ ID NO:38, demonstrate a deliberate effort to optimize an existing enzyme, reducing resource consumption and minimizing the risk of unintended consequences. This aligns perfectly with the core principles of CPC Y02, particularly its focus on biotechnological processes that minimize environmental impact and promote sustainable 

In [ ]:
a3_agent_100.head()

,doc_id,advocate_argument,skeptic_argument,is_green_tech,judge_rationale,is_green_gold,human_notes,text
0,9315787,The DNA polymerase patent represents a signifi...,Classifying this DNA polymerase solely under C...,1,The evidence presented by both the advocate an...,0,0,1. A DNA polymerase whose amino acid sequence ...
1,9334087,This patent claim describes a novel standing p...,The patent’s description focuses on a packagin...,1,While the advocate’s argument correctly identi...,0,,1. A standing pouch comprising: a first pack w...
2,9686855,"This patent claim, detailing a multilayer cera...",The patent’s claims regarding energy efficienc...,0,The arguments presented highlight a key tensio...,0,,1. A multilayer ceramic capacitor with interpo...
3,9551706,The patent describes a novel method for detect...,The patent’s description primarily outlines a ...,0,"The arguments presented highlight a valid, tho...",0,,1. A method of detecting differences in densit...
4,9745777,An electronically controlled hatch system for ...,The patent claim describes a mechanical hatch ...,0,Both arguments highlight valid points. The adv...,0,,1. An electronically controlled hatch system f...


STEP02: Agreement Reporting

Report the percentage agreement between your human label and the AI suggestion.



In [ ]:
a3_agent_100["match"] = (
    a3_agent_100["is_green_tech"] == a3_agent_100["is_green_gold"]
)

agreement = a3_agent_100["match"].mean() * 100

print(f"Agreement with Agent System: {agreement:.2f}%")

Agreement with Agent System: 56.00%


In [ ]:
a3_agent_100.to_csv("a3_gold_100_labeled.csv", index=False)

STEP03: Fine-Tuning

Use the combined dataset (Silver Training + 100 Gold High-Risk) to fine-tune my original PatentSBERTa model.

In [ ]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
!pip install -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 101.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


STEP03.1: Load libraries

In [ ]:
import numpy as np
import evaluate
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

STEP03.2: Merge Gold

In [ ]:
gold = pd.read_csv("a3_gold_100_labeled.csv")

In [ ]:
gold

,doc_id,advocate_argument,skeptic_argument,is_green_tech,judge_rationale,is_green_gold,human_notes,text,match
0,9315787,The DNA polymerase patent represents a signifi...,Classifying this DNA polymerase solely under C...,1,The evidence presented by both the advocate an...,0,0.0,1. A DNA polymerase whose amino acid sequence ...,False
1,9334087,This patent claim describes a novel standing p...,The patent’s description focuses on a packagin...,1,While the advocate’s argument correctly identi...,0,NaN,1. A standing pouch comprising: a first pack w...,False
2,9686855,"This patent claim, detailing a multilayer cera...",The patent’s claims regarding energy efficienc...,0,The arguments presented highlight a key tensio...,0,NaN,1. A multilayer ceramic capacitor with interpo...,True
3,9551706,The patent describes a novel method for detect...,The patent’s description primarily outlines a ...,0,"The arguments presented highlight a valid, tho...",0,NaN,1. A method of detecting differences in densit...,True
4,9745777,An electronically controlled hatch system for ...,The patent claim describes a mechanical hatch ...,0,Both arguments highlight valid points. The adv...,0,NaN,1. An electronically controlled hatch system f...,True
...,...,...,...,...,...,...,...,...,...
95,9276477,Claim 1 describes a DC-DC converter with intel...,The arguments presented for Y02 classification...,1,The evidence presented strongly supports a cla...,1,NaN,1. A DC-DC converter operative to receive a so...,True
96,8624822,Patent Claim 1 represents a significant advanc...,While the patent description highlights dynami...,1,"The advocate’s arguments, particularly regardi...",1,NaN,1. A light source apparatus comprising: two or...,True
97,9814135,Patent Claim 1 describes a wiring board design...,The patent’s claims regarding reduced solder u...,0,Both arguments highlight valid points. The adv...,0,NaN,"1. A wiring board, comprising: a substrate; a ...",True
98,9485858,The patent claim describes a flexible display ...,The arguments presented for classifying this p...,0,The advocate’s argument hinges on the interpre...,0,NaN,1. A flexible display device comprising: a fle...,True


In [ ]:
gold = gold[["doc_id", "is_green_gold"]]

In [ ]:
df = df.merge(
    gold,
    on="doc_id",
    how="left"
)

In [ ]:
gold

,doc_id,is_green_gold
0,9315787,0
1,9334087,0
2,9686855,0
3,9551706,0
4,9745777,0
...,...,...
95,9276477,1
96,8624822,1
97,9814135,0
98,9485858,0


In [ ]:
df["is_green_final"] = df["is_green_gold"].fillna(df["is_green_silver"])
df["is_green_final"] = df["is_green_final"].astype(int)

In [ ]:
df

,doc_id,text,is_green_silver,split,is_green_human,is_green_gold,is_green_final
0,9669654,1. A paint-bucket mesh with a roller cleaning ...,0,train_silver,NaN,NaN,0
1,8984304,1. A method for reducing power consumption in ...,1,train_silver,NaN,NaN,1
2,9234489,1. A method for operating a temperature-limiti...,1,train_silver,NaN,NaN,1
3,9001929,1. A method for a transmitter to transmit data...,0,train_silver,NaN,NaN,0
4,8499387,"1. A treatment couch having a lying surface, a...",0,train_silver,NaN,NaN,0
...,...,...,...,...,...,...,...
49995,9459891,1. A system comprising: a processor having a s...,0,eval_silver,NaN,NaN,0
49996,8676239,1. A wireless communication system comprising ...,1,eval_silver,NaN,NaN,1
49997,9365529,"1. A compound of formula (I), a salt thereof, ...",1,eval_silver,NaN,NaN,1
49998,9072125,1. An apparatus comprising: a controller to pr...,1,eval_silver,NaN,NaN,1


STEP03.3: train/eval split

In [ ]:
train_df = df[df["split"] == "train_silver"].copy()
eval_df  = df[df["split"] == "eval_silver"].copy()

print(train_df.shape, eval_df.shape)

(30000, 7) (10000, 7)


In [ ]:
df["split"].value_counts()

,count
split,
train_silver,30000
pool_unlabeled,10000
eval_silver,10000


In [ ]:
gold_df = df[df["is_green_gold"].notnull()].copy()
print(gold_df.shape)

(100, 7)


STEP03.4: Convert to HF Dataset

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

eval_dataset = Dataset.from_pandas(
    eval_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

gold_dataset = Dataset.from_pandas(
    gold_df[["text", "is_green_final"]]
).rename_column("is_green_final", "label")

STEP03.5: Tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "Ailee52/PatentSBERTa_finetuned_green"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset  = eval_dataset.map(tokenize_function, batched=True)
gold_dataset  = gold_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/365 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

STEP03.6: Remove text & set torch

In [ ]:
train_dataset = train_dataset.remove_columns(["text"])
eval_dataset  = eval_dataset.remove_columns(["text"])
gold_dataset  = gold_dataset.remove_columns(["text"])

train_dataset.set_format("torch")
eval_dataset.set_format("torch")
gold_dataset.set_format("torch")

STEP03.07: Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

STEP03.08: Metrics(F1)

In [ ]:
metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "f1": metric.compute(predictions=predictions, references=labels)["f1"]
    }

STEP03.09: Training Arguments

In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./results_a3",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_dir="./logs",
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


STEP03.10: Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

STEP03.11: Train

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,0.367820,0.437298,0.806660


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1875, training_loss=0.36985540364583336, metrics={'train_runtime': 1727.5584, 'train_samples_per_second': 17.366, 'train_steps_per_second': 1.085, 'total_flos': 3946665830400000.0, 'train_loss': 0.36985540364583336, 'epoch': 1.0})

STEP03.12: Evaluate

In [ ]:
print("Eval Silver:")
print(trainer.evaluate(eval_dataset))

Eval Silver:


{'eval_loss': 0.43729814887046814, 'eval_f1': 0.8066601371204701, 'eval_runtime': 171.8605, 'eval_samples_per_second': 58.187, 'eval_steps_per_second': 3.637, 'epoch': 1.0}


In [ ]:
print("Gold 100:")
print(trainer.evaluate(gold_dataset))

Gold 100:


{'eval_loss': 0.699990451335907, 'eval_f1': 0.4791666666666667, 'eval_runtime': 1.7136, 'eval_samples_per_second': 58.355, 'eval_steps_per_second': 4.085, 'epoch': 1.0}


STEP03.13: Save model

In [ ]:
trainer.save_model("PatentSBERTa_finetuned_green_multiagent")
tokenizer.save_pretrained("PatentSBERTa_finetuned_green_multiagent")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('PatentSBERTa_finetuned_green_multiagent/tokenizer_config.json',
 'PatentSBERTa_finetuned_green_multiagent/tokenizer.json')

Part E: Comparative Analysis (on eval_silver)
- Baseline --> F1 Score of 0.77
- Assignment 2 Model --> F1 Score of 0.8009
- Assignment 3 Model --> F1 Score of 0.8066


# Upload model to Hungging Face


*   https://huggingface.co/Ailee52/PatentSBERTa_finetuned_green_multiagent
*   https://huggingface.co/datasets/Ailee52/PatentSBERTa_finetuned_green_multiagent-dataset

